In [1]:
# Import the custom environment and database settings from another notebook
%run "00_globals_and_db.ipynb"


In [2]:
# FetcherSession: A tool to "visit" websites and download their content
from scrapling.fetchers import FetcherSession
# re: The "Regular Expressions" library used for finding patterns (like dates) in text
import re

# The main website address (base) and the specific page we want to start on (root)
VVTAT_BASE = "https://vvtat.lrv.lt"
ROOT_INDEX_URL = (
    "https://vvtat.lrv.lt/lt/valstybines-vartotoju-teisiu-apsaugos-tarnybos-komisijos-nutarimai/"
    "gincu-sprendimas-ne-eismo-tvarka/"
)

# A pattern to find 4-digit years starting with "20" (e.g., 2023, 2024)
# \b means "boundary" (to avoid catching 12024), \d{2} means two digits
YEAR_RE = re.compile(r"\b(20\d{2})\b")

In [3]:
# Start a "session" to browse the web. 
# Timeout=30 means wait 30 seconds before giving up; retries=3 means try again if it fails.
with FetcherSession(timeout=30, retries=3) as session:
    root = session.get(ROOT_INDEX_URL)

# Print the status code (e.g., 200 means "Success", 404 means "Not Found")
print(root.status)

[2026-04-07 19:17:13] INFO: Fetched (200) <GET https://vvtat.lrv.lt/lt/valstybines-vartotoju-teisiu-apsaugos-tarnybos-komisijos-nutarimai/gincu-sprendimas-ne-eismo-tvarka/> (referer: https://www.google.com/search?q=lrv)


200


In [ ]:
# A dictionary to store the years we find, using the year as the "key" to avoid duplicates
years = {}

# Look at every 'a' tag (HTML links) that has an 'href' (a URL address)
for a in root.css("a[href]"):
    href = a.attrib.get("href", "") # Get the link URL
    txt = norm_text(a.text)         # Get the clickable text of the link (cleaned up)

    # Check if the year is mentioned in the link text
    m_txt = YEAR_RE.search(txt)
    # Check if the year is mentioned inside the URL itself (as a fallback)
    m_href = YEAR_RE.search(href)

    # Filtering Logic: We only want links related to "dispute decisions"
    # If the specific phrase isn't in the URL or the text, we check if we should skip it
    if "gincu-sprendimas-ne-eismo-tvarka" not in href and "gincu-sprendimas-ne-eismo-tvarka" not in txt.lower():
        # If there's no year found in text or link, skip this link entirely
        if not m_txt and not m_href:
            continue

    y = None
    # If the text is exactly a year (like "2024"), use that
    if m_txt and txt.strip() == m_txt.group(1):
        y = int(m_txt.group(1))
    # Otherwise, if the URL contains a year, use that
    elif m_href:
        y = int(m_href.group(1))

    # If we successfully found a year, save the data
    if y:
        # urljoin combines the base site (vvtat.lrv.lt) with the link (like /page1) 
        # to make a full clickable link
        full = urljoin(VVTAT_BASE, href)
        years[y] = {"year": y, "url": full}

# Convert our dictionary into a list, sorted from newest year to oldest
years_list = [years[y] for y in sorted(years.keys(), reverse=True)]

# Define where to save the file (DIR_RAW is defined in the %run notebook)
out_path = DIR_RAW / "years.json"

# Write the list to a file in JSON format
# indent=2 makes the file look "pretty" and readable for humans
out_path.write_text(json.dumps(years_list, ensure_ascii=False, indent=2), encoding="utf-8")

# Output how many years were found and where the file was saved
print(len(years_list), str(out_path))
print (years)

11 c:\Users\Rokas\Desktop\MYSQL-PYTHON-DATA\data_raw\years.json
{2026: {'year': 2026, 'url': 'https://vvtat.lrv.lt/lt/valstybines-vartotoju-teisiu-apsaugos-tarnybos-komisijos-nutarimai/gincu-sprendimas-ne-eismo-tvarka/2026-01/'}, 2025: {'year': 2025, 'url': 'https://vvtat.lrv.lt/lt/valstybines-vartotoju-teisiu-apsaugos-tarnybos-komisijos-nutarimai/gincu-sprendimas-ne-eismo-tvarka/2025/'}, 2024: {'year': 2024, 'url': 'https://vvtat.lrv.lt/lt/valstybines-vartotoju-teisiu-apsaugos-tarnybos-komisijos-nutarimai/gincu-sprendimas-ne-eismo-tvarka/2024/'}, 2023: {'year': 2023, 'url': 'https://vvtat.lrv.lt/lt/valstybines-vartotoju-teisiu-apsaugos-tarnybos-komisijos-nutarimai/gincu-sprendimas-ne-eismo-tvarka/2023/'}, 2022: {'year': 2022, 'url': 'https://vvtat.lrv.lt/lt/valstybines-vartotoju-teisiu-apsaugos-tarnybos-komisijos-nutarimai/gincu-sprendimas-ne-eismo-tvarka/2022/'}, 2021: {'year': 2021, 'url': 'https://vvtat.lrv.lt/lt/valstybines-vartotoju-teisiu-apsaugos-tarnybos-komisijos-nutarimai/gi